In [1]:
#import all needed libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate

In [2]:
#load splitted data from /preprocessed_data
X_train = pd.read_csv('../preprocessed_data/X_train.csv')
y_train = pd.read_csv('../preprocessed_data/y_train.csv').squeeze()
X_test = pd.read_csv('../preprocessed_data/X_test.csv')
y_test = pd.read_csv('../preprocessed_data/y_test.csv').squeeze()

In [3]:
#before training, use only these features:# Enforce final optimized feature set
final_features = [
    'latitude', 'longitude', #'year',
    'month_sin', 'month_cos', 
    #'dow_sin', 'dow_cos',
    'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'wind_magnitude',
    'NO2',
     'CO', 'O3', 
     'AOD', 
    #'pollution_index',
    #'pm25_lag1', 
    #'temp_lag1', #'pollution_lag1',
    # 'aod_pollution_interaction', 'temp_pollution_interaction',
    # 'pressure_pollution_interaction', 'temp_pressure_interaction',
    'building_density', 'road_density_km', 'industrial_presence', 
        'green_space_fraction', 'relative_humidity',

]

# Filter to available features to avoid potential missing column errors
final_features = [f for f in final_features if f in X_train.columns]


X_train = X_train[final_features]
X_test = X_test[final_features]

In [4]:
#train linear regression model and evaluate it using cross validation
lr_model = LinearRegression()
tscv = TimeSeriesSplit(n_splits=5)

cv_results = cross_validate(lr_model, X_train, y_train, cv=tscv, 
                            scoring=('neg_mean_squared_error', 'r2'), 
                            return_train_score=True)

print("Linear Regression CV Train RMSE:", np.mean(np.sqrt(-cv_results['train_neg_mean_squared_error'])))
print("Linear Regression CV Val RMSE:", np.mean(np.sqrt(-cv_results['test_neg_mean_squared_error'])))
print("Linear Regression CV Train R2:", np.mean(cv_results['train_r2']))
print("Linear Regression CV Val R2:", np.mean(cv_results['test_r2']))

#print feature importance for linear regression in a readable format
lr_model.fit(X_train, y_train)
feature_importance = pd.Series(lr_model.coef_, index=X_train.columns).sort_values(ascending=False)
print("Linear Regression Feature Importance:")
print(feature_importance)


Linear Regression CV Train RMSE: 13.55939820956365
Linear Regression CV Val RMSE: 14.76473006586823
Linear Regression CV Train R2: 0.09112934451463499
Linear Regression CV Val R2: -0.03476269578511397
Linear Regression Feature Importance:
month_cos               2.051703
NO2                     1.168269
month_sin               1.031720
building_density        0.911392
pressure_mb             0.872780
AOD                     0.870740
temperature_celsius     0.790253
road_density_km         0.740106
CO                      0.432607
longitude               0.252070
industrial_presence     0.227071
wind_v                  0.224233
relative_humidity       0.212050
wind_u                  0.153505
wind_magnitude         -0.215555
green_space_fraction   -0.649138
O3                     -0.691493
latitude               -0.945472
dtype: float64


In [5]:
#train random forest model and evaluate it using cross validation
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
tscv = TimeSeriesSplit(n_splits=5)

cv_results = cross_validate(rf_model, X_train, y_train, cv=tscv, 
                            scoring=('neg_mean_squared_error', 'r2'), 
                            return_train_score=True)

print("Random Forest CV Train RMSE:", np.mean(np.sqrt(-cv_results['train_neg_mean_squared_error'])))
print("Random Forest CV Val RMSE:", np.mean(np.sqrt(-cv_results['test_neg_mean_squared_error'])))
print("Random Forest CV Train R2:", np.mean(cv_results['train_r2']))
print("Random Forest CV Val R2:", np.mean(cv_results['test_r2']))

#print feature importance for random forest in a readable format
rf_model.fit(X_train, y_train)
feature_importance = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Random Forest Feature Importance:")
print(feature_importance)

Random Forest CV Train RMSE: 5.016262762867782
Random Forest CV Val RMSE: 17.73175801755027
Random Forest CV Train R2: 0.8805288703497073
Random Forest CV Val R2: -1.4072157228636066
Random Forest Feature Importance:
temperature_celsius     0.121985
AOD                     0.115234
relative_humidity       0.098562
green_space_fraction    0.082626
CO                      0.081356
latitude                0.068219
wind_u                  0.062084
NO2                     0.056793
wind_magnitude          0.054917
O3                      0.046433
pressure_mb             0.043596
longitude               0.041870
wind_v                  0.032661
road_density_km         0.022725
industrial_presence     0.022349
month_cos               0.022083
building_density        0.016660
month_sin               0.009848
dtype: float64


In [6]:
#train xgboost model and evaluate it using cross validation
xgb_model = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
tscv = TimeSeriesSplit(n_splits=5)

cv_results = cross_validate(xgb_model, X_train, y_train, cv=tscv, 
                            scoring=('neg_mean_squared_error', 'r2'), 
                            return_train_score=True)

print("XGBoost CV Train RMSE:", np.mean(np.sqrt(-cv_results['train_neg_mean_squared_error'])))
print("XGBoost CV Val RMSE:", np.mean(np.sqrt(-cv_results['test_neg_mean_squared_error'])))
print("XGBoost CV Train R2:", np.mean(cv_results['train_r2']))
print("XGBoost CV Val R2:", np.mean(cv_results['test_r2']))


#print feature importance for xgboost in a readable format
xgb_model.fit(X_train, y_train)
feature_importance = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)  
print("XGBoost Feature Importance:")
print(feature_importance)

XGBoost CV Train RMSE: 3.170339127011064
XGBoost CV Val RMSE: 19.632155454696026
XGBoost CV Train R2: 0.9453815286838184
XGBoost CV Val R2: -2.4269248211773347
XGBoost Feature Importance:
month_cos               0.190088
green_space_fraction    0.133759
wind_u                  0.090834
O3                      0.074521
relative_humidity       0.053624
pressure_mb             0.049744
longitude               0.049367
temperature_celsius     0.048753
latitude                0.046108
wind_v                  0.043487
wind_magnitude          0.038570
NO2                     0.033608
CO                      0.032699
building_density        0.031474
AOD                     0.030874
industrial_presence     0.020909
month_sin               0.019704
road_density_km         0.011879
dtype: float32
